# L1 and L2 regularisation in a neural network
Why a bigger network overfits more, then the same 128-128 network on `make_moons` without regularisation,
with L2 and with L1: decision boundaries, training curves and the first-layer weights.
Results go to `data/` for the figures in `images/`.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"      # hide TensorFlow's start-up messages
import numpy as np
import pandas as pd
import keras
import tensorflow as tf
from sklearn.datasets import make_moons

tf.config.experimental.enable_op_determinism()   # same numbers on every run
os.makedirs("data", exist_ok=True)

## The data
100 points in two noisy half-moons.

In [ ]:
X, y = make_moons(n_samples=100, noise=0.25, random_state=2)
pd.DataFrame({"x1": X[:, 0], "x2": X[:, 1], "y": y}).round(4).to_csv("data/points.csv", index=False)
xx, yy = np.meshgrid(np.linspace(-2, 3, 151), np.linspace(-1.75, 2.25, 121))
grid = np.c_[xx.ravel(), yy.ravel()]
boundaries = {"x1": grid[:, 0], "x2": grid[:, 1]}

## 1. More neurons, more complex boundaries
One hidden layer with 1, 10, 50 or 1,000 ReLU neurons, trained on all 100 points.

In [ ]:
for n in [1, 10, 50, 1000]:
    keras.utils.set_random_seed(0)
    model = keras.Sequential([keras.Input(shape=(2,)),
                              keras.layers.Dense(n, activation="relu"),
                              keras.layers.Dense(1, activation="sigmoid")])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.01), loss="binary_crossentropy",
                  metrics=["accuracy"])
    model.fit(X, y, epochs=1000, verbose=0)
    _, acc = model.evaluate(X, y, verbose=0)
    print(f"{n:5d} neurons: training accuracy {acc:.2f}")
    boundaries[f"n={n}"] = model.predict(grid, verbose=0).ravel()

## 2. The same network without and with regularisation

In [ ]:
def build(regularizer=None, seed=0):
    keras.utils.set_random_seed(seed)
    model = keras.Sequential([
        keras.Input(shape=(2,)),
        keras.layers.Dense(128, activation="relu", kernel_regularizer=regularizer),
        keras.layers.Dense(128, activation="relu", kernel_regularizer=regularizer),
        keras.layers.Dense(1, activation="sigmoid"),
    ])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.01), loss="binary_crossentropy",
                  metrics=["accuracy"])
    return model

build().summary()

In [ ]:
from keras import regularizers

runs = {"none": None, "L2": regularizers.L2(0.03), "L1": regularizers.L1(0.001)}
histories, weights, scores = {}, {}, []
for name, reg in runs.items():
    model = build(reg)
    h = model.fit(X, y, epochs=2000, validation_split=0.2, verbose=0).history   # last 20 points validate
    histories[f"loss {name}"], histories[f"val_loss {name}"] = h["loss"], h["val_loss"]
    histories[f"acc {name}"], histories[f"val_acc {name}"] = h["accuracy"], h["val_accuracy"]
    W1 = model.get_weights()[0]                  # first layer: shape (2, 128)
    weights[name] = W1.reshape(256)
    scores.append({"regularisation": name, "train_acc": h["accuracy"][-1], "val_acc": h["val_accuracy"][-1],
                   "val_loss": h["val_loss"][-1], "w_min": W1.min(), "w_max": W1.max(),
                   "share_near_0": np.mean(np.abs(W1) < 1e-3)})
    boundaries[name] = model.predict(grid, verbose=0).ravel()
scores = pd.DataFrame(scores)
print(scores.round(3))

## 3. The penalty Keras adds
`L2(0.03)` adds 0.03 times the sum of squared weights of the layer to the loss (no 1/2, no 1/n).

In [ ]:
W = np.array([[0.5, -1.0], [2.0, 0.1]], dtype="float32")
print("L2(0.03):", float(regularizers.L2(0.03)(W)), "= 0.03 *", float((W ** 2).sum()))
print("L1(0.001):", float(regularizers.L1(0.001)(W)), "= 0.001 *", float(np.abs(W).sum()))

## Saving results for the figures

In [ ]:
pd.DataFrame(boundaries).round(4).to_csv("data/boundary_grid.csv", index=False)
pd.DataFrame(histories).round(5).to_csv("data/histories.csv", index_label="epoch")
pd.DataFrame(weights).round(5).to_csv("data/first_layer_weights.csv", index=False)
scores.round(5).to_csv("data/scores.csv", index=False)
print("saved")